# Naive Baseline — Prediksi Arah Kurs USD/IDR (JISDOR)

**Tugas 2, butir 2b**: *"Latih Model Baseline Time-Series menggunakan data historis nilai tukar (misal: ARIMA, XGBoost, atau Naive Baseline)."*

Notebook ini mengevaluasi dua strategi naive baseline:
1. **Persistence of Direction** — prediksi arah besok = arah hari ini
2. **Majority-Class** — selalu prediksi kelas mayoritas di Train (NAIK)

Tujuannya: menetapkan *lower bound* performa. Model selanjutnya (ARIMA, XGBoost, model gabungan NLP) harus **mengalahkan** baseline ini agar dianggap layak.

---

**Kontributor**: Mikail Achmad — 24/542370/PA/23026

In [ ]:
import sys
from pathlib import Path

# Supaya bisa import dari src/
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.models.naive_baseline import (
    predict_persistence,
    predict_majority,
    evaluate_baseline,
    LABEL_ORDER,
)

print(f"Repo root: {REPO_ROOT}")

## 1. Muat Data Split

In [ ]:
SPLIT_DIR = REPO_ROOT / "data" / "split"

train = pd.read_csv(SPLIT_DIR / "train.csv", parse_dates=["tanggal"])
val   = pd.read_csv(SPLIT_DIR / "val.csv",   parse_dates=["tanggal"])
test  = pd.read_csv(SPLIT_DIR / "test.csv",  parse_dates=["tanggal"])

print(f"Train: {len(train)} baris  ({train.tanggal.min().date()} → {train.tanggal.max().date()})")
print(f"Val  : {len(val)} baris  ({val.tanggal.min().date()} → {val.tanggal.max().date()})")
print(f"Test : {len(test)} baris  ({test.tanggal.min().date()} → {test.tanggal.max().date()})")

## 2. Distribusi Kelas Target

Sebelum evaluasi baseline, kita lihat distribusi kelas di setiap split. Ini penting karena:
- Kelas yang *imbalanced* membuat accuracy saja tidak cukup
- Majority-class baseline akan setinggi proporsi kelas terbanyak

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
colors = {"NAIK": "#2ecc71", "STABIL": "#3498db", "TURUN": "#e74c3c"}

for ax, (name, df) in zip(axes, [("Train", train), ("Val", val), ("Test", test)]):
    counts = df["target"].value_counts().reindex(LABEL_ORDER)
    bars = ax.bar(LABEL_ORDER, counts, color=[colors[l] for l in LABEL_ORDER], edgecolor="white", linewidth=0.5)
    ax.set_title(f"{name} (n={len(df)})", fontweight="bold", fontsize=13)
    ax.set_ylabel("Jumlah Hari" if ax == axes[0] else "")
    # Label jumlah dan persentase di atas bar
    for bar, count in zip(bars, counts):
        pct = count / len(df) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f"{count}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=10)
    ax.set_ylim(0, counts.max() * 1.25)

fig.suptitle("Distribusi Kelas Target per Split", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(REPO_ROOT / "results" / "target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Evaluasi Naive Baseline

In [ ]:
results = {}

for split_name, split_df in [("val", val), ("test", test)]:
    y_true = split_df["target"].values

    # 1) Persistence of Direction
    y_pred_persist = predict_persistence(split_df["target"])
    res_persist = evaluate_baseline(y_true, y_pred_persist, "Persistence of Direction")

    # 2) Majority-Class
    y_pred_majority = predict_majority(train["target"], len(split_df))
    res_majority = evaluate_baseline(y_true, y_pred_majority, "Majority-Class")

    results[split_name] = {
        "persistence": res_persist,
        "majority_class": res_majority,
    }

    # Tampilkan ringkasan
    print(f"\n{'='*50}")
    print(f"  Split: {split_name.upper()}")
    print(f"{'='*50}")
    for key, res in [("Persistence", res_persist), ("Majority", res_majority)]:
        print(f"  {key:<15} Accuracy={res['accuracy']:.4f}  Macro-F1={res['macro_f1']:.4f}")

## 4. Confusion Matrix — Visualisasi

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

plot_configs = [
    (0, 0, "val",  "persistence",    "Persistence — Val"),
    (0, 1, "val",  "majority_class", "Majority-Class — Val"),
    (1, 0, "test", "persistence",    "Persistence — Test"),
    (1, 1, "test", "majority_class", "Majority-Class — Test"),
]

for row, col, split, strategy, title in plot_configs:
    ax = axes[row][col]
    cm = np.array(results[split][strategy]["confusion_matrix"])
    
    # Normalize per-row (recall)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    
    # Annotate cells
    for i in range(len(LABEL_ORDER)):
        for j in range(len(LABEL_ORDER)):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.0%})",
                    ha="center", va="center", fontsize=10, color=color, fontweight="bold")
    
    ax.set_xticks(range(len(LABEL_ORDER)))
    ax.set_yticks(range(len(LABEL_ORDER)))
    ax.set_xticklabels(LABEL_ORDER)
    ax.set_yticklabels(LABEL_ORDER)
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
    acc = results[split][strategy]["accuracy"]
    f1 = results[split][strategy]["macro_f1"]
    ax.set_title(f"{title}\nAcc={acc:.2%}  F1={f1:.2%}", fontweight="bold", fontsize=11)

fig.suptitle("Confusion Matrix — Naive Baseline", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(REPO_ROOT / "results" / "naive_baseline_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Perbandingan Metrik

In [ ]:
# Bar chart perbandingan
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, metric, label in zip(axes, ["accuracy", "macro_f1"], ["Accuracy", "Macro F1-Score"]):
    strategies = ["Persistence", "Majority-Class"]
    val_scores = [
        results["val"]["persistence"][metric],
        results["val"]["majority_class"][metric],
    ]
    test_scores = [
        results["test"]["persistence"][metric],
        results["test"]["majority_class"][metric],
    ]

    x = np.arange(len(strategies))
    width = 0.3

    bars1 = ax.bar(x - width/2, val_scores, width, label="Val", color="#3498db", edgecolor="white")
    bars2 = ax.bar(x + width/2, test_scores, width, label="Test", color="#e67e22", edgecolor="white")

    for bars in [bars1, bars2]:
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{bar.get_height():.2%}", ha="center", va="bottom", fontsize=10, fontweight="bold")

    ax.set_ylabel(label)
    ax.set_title(label, fontweight="bold", fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(strategies)
    ax.set_ylim(0, 0.6)
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

fig.suptitle("Perbandingan Naive Baseline", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(REPO_ROOT / "results" / "naive_baseline_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Analisis & Kesimpulan

### Temuan Utama

| Strategi | Split | Accuracy | Macro F1 |
|----------|-------|----------|----------|
| Persistence of Direction | Val | ~37.8% | ~35.4% |
| Persistence of Direction | Test | ~38.0% | ~36.4% |
| Majority-Class (NAIK) | Val | ~36.1% | ~17.7% |
| Majority-Class (NAIK) | Test | ~44.6% | ~20.6% |

### Interpretasi

1. **Persistence of Direction** konsisten di Val dan Test (Acc ≈38%, F1 ≈35-36%), menandakan *tidak ada* tren inertia yang kuat pada pasar forex USD/IDR. Ini wajar karena pasar valuta asing cenderung efisien — pergerakan harian bersifat *near-random-walk*.

2. **Majority-Class** punya accuracy sedikit lebih tinggi di Test (44.6%) karena distribusi kelas NAIK lebih besar (44.6% di test), tapi Macro F1-nya sangat rendah (20.6%) karena recall kelas STABIL dan TURUN = 0%.

3. **Macro F1 lebih informatif** daripada Accuracy untuk kasus ini karena kelas imbalanced. Persistence (F1 ≈36%) jelas lebih seimbang daripada Majority (F1 ≈18-21%).

### Implikasi untuk Model Selanjutnya

- **Lower bound yang harus dikalahkan**: Macro F1 ≈ 36% (Persistence) di Val/Test.
- Model ARIMA/XGBoost (time-series murni) dan model gabungan (time-series + fitur NLP) harus menghasilkan Macro F1 > 36% agar dianggap memberikan nilai tambah.
- Distribusi kelas yang tidak terlalu ekstrem (NAIK ≈40%, STABIL ≈30%, TURUN ≈30%) membuat masalah ini *feasible* — random baseline ≈ 33%.

In [ ]:
# Simpan ringkasan hasil
summary = {
    "baseline": "Naive Baseline",
    "strategies": ["Persistence of Direction", "Majority-Class"],
    "lower_bound_metric": "Macro F1-Score",
    "lower_bound_value": max(
        results["test"]["persistence"]["macro_f1"],
        results["test"]["majority_class"]["macro_f1"]
    ),
    "results": results,
}

with open(REPO_ROOT / "results" / "naive_baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Hasil tersimpan di results/naive_baseline_results.json")
print(f"\nLower bound untuk model selanjutnya: Macro F1 > {summary['lower_bound_value']:.2%}")